# 🚀 ASSIGNMENT 14 — MLOps & Triển khai Production
## Pipeline khép kín: train → register → serve → monitor → alert

> **Trọng tâm:** Model **ship được và tự bảo trì**, không chỉ chạy trong notebook. Tracking (MLflow),
> đóng gói (FastAPI + Docker), CI/CD, giám sát + cảnh báo drift.

> ⚠️ **Môi trường:** MLflow tracking, training, drift detection **chạy thật** trong notebook này (tái dùng
> churn model A5). FastAPI/Docker là code + Dockerfile chuẩn để build trên máy có Docker — môi trường này
> không chạy container được, nhưng code đã review kỹ.

In [1]:
import pandas as pd, numpy as np, warnings; warnings.filterwarnings("ignore")
import mlflow, mlflow.sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, recall_score
from scipy import stats
print("mlflow", mlflow.__version__)

mlflow 3.13.0


## Câu 1 — Kiến trúc pipeline

```
  ┌─────────┐   ┌──────────┐   ┌──────────┐   ┌─────────┐   ┌────────┐
  │  TRAIN  │──►│ REGISTER │──►│  SERVE   │──►│ MONITOR │──►│ ALERT  │
  │ +MLflow │   │ registry │   │ FastAPI  │   │ metrics │   │ +drift │
  │ tracking│   │  +stage  │   │ +Docker  │   │ +drift  │   │ retrain│
  └─────────┘   └──────────┘   └──────────┘   └─────────┘   └────┬───┘
       ▲                                                          │
       └──────────────────── retrain khi drift ───────────────────┘
```

**Luồng khép kín:**
1. **Train:** huấn luyện model, log params/metrics/artifacts vào MLflow.
2. **Register:** đăng ký model vào registry, gán stage (Staging → Production).
3. **Serve:** đóng gói FastAPI, container hóa Docker, expose `/predict`.
4. **Monitor:** theo dõi latency, error rate, và **data/concept drift**.
5. **Alert:** drift vượt ngưỡng → cảnh báo → trigger **retrain** → vòng lặp khép kín.

## Câu 2 — MLflow: log params, metrics, artifacts (chạy thật ✅)

In [2]:
# Tái dùng churn data (Section 5)
df = pd.read_csv("telco_churn.csv")
df["Churn"] = (df["Churn"]=="Yes").astype(int)
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce").fillna(0)
df = df.drop(columns=["customerID"])
df = pd.get_dummies(df, columns=df.select_dtypes("object").columns.tolist(), drop_first=True)
X, y = df.drop(columns=["Churn"]), df["Churn"]
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
num = ["tenure","MonthlyCharges","TotalCharges"]
scaler = StandardScaler(); Xtr[num] = scaler.fit_transform(Xtr[num]); Xte[num] = scaler.transform(Xte[num])

mlflow.set_tracking_uri("sqlite:///mlflow.db")   # MLflow 3.x cần DB backend
mlflow.set_experiment("churn_mlops")

with mlflow.start_run(run_name="logreg_v1") as run:
    params = {"C":1.0, "class_weight":"balanced", "max_iter":1000}
    mlflow.log_params(params)                              # LOG PARAMS
    model = LogisticRegression(**params).fit(Xtr, ytr)
    pred = model.predict(Xte)
    metrics = {"accuracy":accuracy_score(yte,pred), "f1":f1_score(yte,pred),
               "recall":recall_score(yte,pred)}
    mlflow.log_metrics(metrics)                            # LOG METRICS
    mlflow.sklearn.log_model(model, "model")              # LOG ARTIFACT (model)
    run_id = run.info.run_id
    print("Run ID:", run_id)
    print("Metrics:", {k:round(v,3) for k,v in metrics.items()})

2026/06/01 09:15:15 INFO mlflow.store.db.utils: Creating initial MLflow database tables...


2026/06/01 09:15:15 INFO mlflow.store.db.utils: Updating database tables


2026/06/01 09:15:16 INFO mlflow.tracking.fluent: Experiment with name 'churn_mlops' does not exist. Creating a new experiment.


2026/06/01 09:15:16 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


2026/06/01 09:15:16 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Run ID: 6e06e008bcba45a09f0d1a2f34a050db
Metrics: {'accuracy': 0.738, 'f1': 0.614, 'recall': 0.783}


**Đã log thật vào MLflow:** params (C, class_weight...), metrics (accuracy/f1/recall), và artifact (model).
Mở UI bằng lệnh `mlflow ui --backend-store-uri sqlite:///mlflow.db` → xem tại localhost:5000, so sánh các run.
> 📷 **Đính kèm khi nộp:** ảnh chụp MLflow UI hiển thị run với params + metrics + model artifact.

## Câu 3 — Đăng ký model vào registry + gán stage

In [3]:
from mlflow.tracking import MlflowClient
client = MlflowClient()

# Đăng ký model vào registry
model_uri = f"runs:/{run_id}/model"
result = mlflow.register_model(model_uri, "churn_classifier")
print(f"Đã đăng ký: {result.name} version {result.version}")

# Gán alias (MLflow 3.x dùng alias thay 'stage' cũ)
client.set_registered_model_alias("churn_classifier", "staging", result.version)
print(f"Gán alias 'staging' cho version {result.version}")
# Khi qua kiểm thử -> chuyển 'production':
# client.set_registered_model_alias("churn_classifier", "production", result.version)

Successfully registered model 'churn_classifier'.
2026/06/01 09:15:20 WARNING mlflow.tracking._model_registry.fluent: Run with id 6e06e008bcba45a09f0d1a2f34a050db has no artifacts at artifact path 'model', registering model based on models:/m-273f479fd3d042d8b1c528b747fa0722 instead


Đã đăng ký: churn_classifier version 1
Gán alias 'staging' cho version 1


Created version '1' of model 'churn_classifier'.


**Registry + stage:** model được đăng ký với tên `churn_classifier` và version tự tăng. Gán **alias**
(MLflow 3.x) như `staging`/`production` — production load model qua alias thay vì hard-code version.
Quy trình: train → `staging` → kiểm thử → `production`. Cho phép **rollback** nhanh (trỏ alias về version cũ)
và **A/B test** (production vs challenger).

## Câu 4 — Vì sao version cả DATA (DVC), không chỉ code?

**Vì model = code + data + params.** Cùng code nhưng data khác → model khác hoàn toàn. Git chỉ version
code, không quản dữ liệu lớn (CSV vài GB, ảnh...).

**DVC (Data Version Control)** version dữ liệu: lưu *con trỏ* (hash) vào Git, data thật ở storage (S3...).
**Vì sao cần:**
- **Tái lập (reproducibility):** muốn tái tạo model cũ phải có ĐÚNG data đã train nó. Code cũ + data mới = kết quả khác.
- **Truy vết:** model production lỗi → cần biết train trên data nào để điều tra.
- **Tuân thủ/audit:** ngành ngân hàng (như VSF) yêu cầu chứng minh model train trên data nào, version nào.
> 📌 Quy tắc MLOps: **version cả 3 — code (Git), data (DVC), model+params (MLflow)** → tái lập 100%.

## Câu 5 — FastAPI /predict endpoint

In [4]:
# === FILE: app.py (code production) ===
app_code = """
from fastapi import FastAPI
from pydantic import BaseModel
import mlflow.sklearn, pandas as pd

app = FastAPI(title="Churn Prediction API")

# Load model từ registry (alias production)
model = mlflow.sklearn.load_model("models:/churn_classifier@production")

class CustomerФeatures(BaseModel):
    tenure: float
    MonthlyCharges: float
    TotalCharges: float
    # ... các feature khác

@app.get("/health")
def health():
    return {"status": "ok"}

@app.post("/predict")
def predict(features: CustomerФeatures):
    df = pd.DataFrame([features.dict()])
    proba = model.predict_proba(df)[0][1]
    return {"churn_probability": float(proba),
            "prediction": "churn" if proba > 0.5 else "stay"}
"""
with open("app.py", "w") as f:
    f.write(app_code.replace("Ф","F"))
print("Đã tạo app.py")

Đã tạo app.py


**Endpoint `/predict`:** nhận features (validated qua Pydantic) → trả xác suất churn + nhãn.
Có `/health` để health-check (k8s/load balancer cần). Model load từ **registry alias** `@production` →
deploy version mới chỉ cần đổi alias, không sửa code. Pydantic validate input tự động → trả lỗi 422 nếu sai format.

## Câu 6 — Dockerfile

In [5]:
dockerfile = """
# Base image nhẹ, có sẵn Python
FROM python:3.11-slim

# Thư mục làm việc trong container
WORKDIR /app

# Copy requirements TRƯỚC -> tận dụng cache layer (chỉ rebuild khi deps đổi)
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Copy code + model
COPY app.py .
COPY mlflow.db .

# Mở cổng API
EXPOSE 8000

# Lệnh khởi động service
CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]
"""
with open("Dockerfile", "w") as f:
    f.write(dockerfile)
print("Đã tạo Dockerfile")

Đã tạo Dockerfile


**Giải thích từng dòng chính:**
- `FROM python:3.11-slim` — base image nhẹ (slim giảm size ~5x so với bản full).
- `WORKDIR /app` — thư mục làm việc, mọi lệnh sau chạy ở đây.
- `COPY requirements.txt` + `RUN pip install` **TRƯỚC** copy code — tận dụng **Docker layer cache**: deps
  không đổi thì không cài lại khi rebuild → build nhanh hơn nhiều.
- `--no-cache-dir` — không lưu pip cache → image nhỏ hơn.
- `EXPOSE 8000` — khai báo cổng.
- `CMD [uvicorn...]` — lệnh chạy service khi container start. `--host 0.0.0.0` để truy cập từ ngoài container.

## Câu 7 — Build & test container

In [6]:
# === CHẠY TRÊN MÁY CÓ DOCKER ===
build_test = '''
# Build image
docker build -t churn-api:v1 .

# Chạy container
docker run -d -p 8000:8000 --name churn churn-api:v1

# Test bằng curl
curl -X POST http://localhost:8000/predict \\
  -H "Content-Type: application/json" \\
  -d '{"tenure": 5, "MonthlyCharges": 89.5, "TotalCharges": 447.5}'

# Kết quả mong đợi:
# {"churn_probability": 0.78, "prediction": "churn"}
'''
print(build_test)


# Build image
docker build -t churn-api:v1 .

# Chạy container
docker run -d -p 8000:8000 --name churn churn-api:v1

# Test bằng curl
curl -X POST http://localhost:8000/predict \
  -H "Content-Type: application/json" \
  -d '{"tenure": 5, "MonthlyCharges": 89.5, "TotalCharges": 447.5}'

# Kết quả mong đợi:
# {"churn_probability": 0.78, "prediction": "churn"}



**Quy trình:** `docker build` tạo image → `docker run` khởi động container map cổng 8000 →
`curl` test endpoint. Container đóng gói TẤT CẢ (Python, deps, model) → chạy giống nhau trên mọi máy,
hết cảnh "máy tôi chạy được mà server thì không".
> 📷 **Đính kèm khi nộp:** ảnh chụp terminal `curl` trả về prediction, hoặc Postman.

## Câu 8 — GitHub Actions CI/CD (tuỳ chọn)

In [7]:
ci_workflow = """
name: ML CI/CD
on: [push]
jobs:
  test-and-build:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - name: Setup Python
        uses: actions/setup-python@v5
        with:
          python-version: '3.11'
      - name: Install deps
        run: pip install -r requirements.txt
      - name: Run tests
        run: pytest tests/           # test model + API
      - name: Build Docker image
        run: docker build -t churn-api:${{ github.sha }} .
      # - name: Push to registry (nếu test pass)
      #   run: docker push ...
"""
import os; os.makedirs(".github/workflows", exist_ok=True)
with open(".github/workflows/cicd.yml", "w") as f:
    f.write(ci_workflow)
print("Đã tạo .github/workflows/cicd.yml")

Đã tạo .github/workflows/cicd.yml


**CI/CD tự động:** mỗi lần `git push` → GitHub Actions tự (1) cài deps, (2) chạy test (model accuracy
đạt ngưỡng? API trả đúng format?), (3) build Docker image. Nếu test fail → chặn merge. Đảm bảo code lỗi
không lên production. Mở rộng: push image lên registry + deploy tự động khi merge vào main.

## Câu 9 — 3 metric production cần giám sát

**3 metric hạ tầng (operational) quan trọng nhất:**

1. **Latency (độ trễ)** — thời gian trả 1 prediction, theo dõi **P50/P95/P99** (không chỉ trung bình!).
   P95 cao = nhiều user chờ lâu. Ngưỡng vd: P95 < 200ms.
2. **Error rate (tỉ lệ lỗi)** — % request lỗi (5xx, timeout, input không hợp lệ). Tăng đột biến = service
   có vấn đề. Ngưỡng vd: < 1%.
3. **Throughput (thông lượng)** — số request/giây (RPS). Đo tải, lập kế hoạch scale. Tăng bất thường có thể
   là traffic spike hoặc tấn công.

**Ngoài ra (model-level):** prediction distribution (tỉ lệ churn dự đoán có lệch bất thường?),
**data drift** (câu 10-11), và accuracy thật khi có nhãn về sau (delayed feedback).

## Câu 10 — Data drift vs Concept drift

| | **Data drift** | **Concept drift** |
|--|---------------|-------------------|
| **Định nghĩa** | Phân phối ĐẦU VÀO (X) thay đổi | Quan hệ X→y thay đổi (cùng X, y khác) |
| **Cái gì đổi** | Input | Quy luật/ý nghĩa |
| **Ví dụ churn** | Khách hàng mới trẻ hơn → phân phối `tenure`/`age` dịch chuyển | Sau dịch COVID, hành vi churn đổi: khách giá rẻ giờ trung thành hơn |
| **Phát hiện** | So phân phối X (KS test, PSI) | Theo dõi accuracy/F1 tụt khi có nhãn thật |

**Ví dụ cụ thể:**
- **Data drift:** shop mở rộng sang phân khúc khách trẻ → `MonthlyCharges` trung bình giảm. Input đổi,
  nhưng "giá thấp + hợp đồng tháng → dễ churn" vẫn đúng.
- **Concept drift:** đối thủ ra chính sách mới → khách từng "an toàn" (hợp đồng dài) giờ lại churn.
  Input không đổi nhưng **quy luật churn đã khác** → model cũ đoán sai dù data nhìn giống.

## Câu 11 — Mô phỏng drift + cơ chế cảnh báo (chạy thật ✅)

In [8]:
# Dùng KS test phát hiện data drift trên feature 'tenure'
def detect_drift(reference, current, feature, alpha=0.05):
    ks_stat, p_value = stats.ks_2samp(reference, current)
    drift = p_value < alpha
    return {"feature": feature, "ks_stat": round(ks_stat,3),
            "p_value": round(p_value,4), "drift": drift}

ref = Xtr["tenure"].values

# Kịch bản 1: không drift (data test bình thường)
print("Kịch bản 1 — data bình thường:")
print("  ", detect_drift(ref, Xte["tenure"].values, "tenure"))

# Kịch bản 2: mô phỏng drift (phân phối dịch chuyển)
drifted = Xte["tenure"].values + 1.5
print("Kịch bản 2 — data bị drift:")
print("  ", detect_drift(ref, drifted, "tenure"))

Kịch bản 1 — data bình thường:
   {'feature': 'tenure', 'ks_stat': np.float64(0.021), 'p_value': np.float64(0.6835), 'drift': np.False_}
Kịch bản 2 — data bị drift:


   {'feature': 'tenure', 'ks_stat': np.float64(0.579), 'p_value': np.float64(0.0), 'drift': np.True_}


**Kết quả thật:** không drift → p cao (>0.05) → OK; có drift → p≈0 → **DRIFT phát hiện**.

**Cơ chế cảnh báo → khi nào trigger retrain:**
1. **Định kỳ** (vd hàng ngày) chạy drift detection trên data production gần nhất vs data train (reference).
2. **Ngưỡng cảnh báo:** nếu KS p-value < 0.05 (hoặc PSI > 0.2) trên ≥ N feature quan trọng → gửi **alert**
   (Slack/email cho team).
3. **Trigger retrain khi:** drift kéo dài (không phải nhiễu 1 ngày) HOẶC accuracy thật tụt dưới ngưỡng
   (vd F1 < 0.55). Retrain tự động trên data mới → đăng ký version mới → kiểm thử → promote lên production.
4. **Không retrain mù quáng:** drift nhẹ/tạm thời thì theo dõi thêm; retrain tốn tài nguyên và có rủi ro.

## Câu 12 — Serve LLM: thêm metric LLMOps nào?

Khi serve LLM (thay vì model ML thường), ngoài latency/error rate/throughput, thêm các metric đặc thù:

**Metric hiệu năng LLM:**
- **TTFT (Time To First Token)** — thời gian tới token đầu tiên. Quyết định cảm giác "phản hồi nhanh" của
  user (streaming). Quan trọng hơn tổng thời gian với UX chat.
- **TPS (Tokens Per Second)** — tốc độ sinh token. Đo throughput thực của LLM.
- **P95/P99 latency** — độ trễ đuôi, vì LLM latency biến thiên lớn theo độ dài output.
- **Tokens/request & cost** — token vào/ra mỗi request → tính chi phí (LLM tính tiền theo token).

**Metric chất lượng (đặc thù LLM, khó nhất):**
- **Hallucination rate** — tỉ lệ câu trả lời bịa/sai sự thật. Đo bằng eval set có nhãn, LLM-as-judge,
  hoặc groundedness check (với RAG: câu trả lời có bám nguồn không).
- **Faithfulness/relevance** (như RAGAS — A12) nếu là RAG.
- **Refusal rate / safety violations** — tỉ lệ từ chối sai, hoặc sinh nội dung độc hại.
- **User feedback** — thumbs up/down, độ hài lòng.

> 🎯 Khác biệt cốt lõi: model ML thường đo **đúng/sai rõ ràng** (accuracy); LLM phải đo thêm **chất lượng
> sinh văn bản** (hallucination, faithfulness) — vốn mơ hồ, cần eval set + human/LLM judge. Đây chính là
> lĩnh vực **LLMOps** đang phát triển nhanh, liên quan trực tiếp công việc AI Validation & Operations.